In [1]:
!pip -q install transformers datasets scikit-learn accelerate
import json, time, collections, numpy as np, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

LABELS = ["bug", "feature", "question", "docs"]
l2i = {l: i for i, l in enumerate(LABELS)}
def load(s): return [json.loads(l) for l in open(f"{s}.jsonl")]
tr, va, te = load("train"), load("val"), load("test")

tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def to_ds(rows):
    ds = Dataset.from_dict({"text": [r["text"] for r in rows],
                            "label": [l2i[r["label"]] for r in rows]})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=256), batched=True)
dtr, dva, dte = to_ds(tr), to_ds(va), to_ds(te)

# class weights: docs is the smallest class, so weight it up
cnt = collections.Counter(r["label"] for r in tr)
weights = torch.tensor([len(tr) / (len(LABELS) * cnt[l]) for l in LABELS], dtype=torch.float)
print("class weights:", dict(zip(LABELS, weights.tolist())))

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=4,
    id2label={i: l for i, l in enumerate(LABELS)}, label2id=l2i)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, labels,
                                                 weight=weights.to(out.logits.device))
        return (loss, out) if return_outputs else loss

def metrics(p):
    pred = p.predictions.argmax(-1)
    return {"accuracy": accuracy_score(p.label_ids, pred),
            "macro_f1": f1_score(p.label_ids, pred, average="macro")}

args = TrainingArguments(output_dir="out", num_train_epochs=3, per_device_train_batch_size=16,
                         per_device_eval_batch_size=32, learning_rate=3e-5,
                         eval_strategy="epoch", save_strategy="epoch",
                         load_best_model_at_end=True, metric_for_best_model="macro_f1",
                         report_to="none", fp16=torch.cuda.is_available())
trainer = WeightedTrainer(model=model, args=args, train_dataset=dtr, eval_dataset=dva,
                          data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()

t0 = time.time()
pred = trainer.predict(dte)
secs = time.time() - t0
y_pred = pred.predictions.argmax(-1); y_true = pred.label_ids
print({"method": "Fine-tuned DistilBERT",
       "accuracy": round(accuracy_score(y_true, y_pred), 4),
       "macro_f1": round(f1_score(y_true, y_pred, average="macro"), 4),
       "sec_per_issue": round(secs / len(y_true), 4), "n": len(y_true)})
print(classification_report(y_true, y_pred, target_names=LABELS))
print("confusion matrix (rows=true, cols=pred, order:", LABELS, ")")
print(confusion_matrix(y_true, y_pred))

trainer.save_model("maintainerbot-distilbert"); tok.save_pretrained("maintainerbot-distilbert")
!zip -rq maintainerbot-distilbert.zip maintainerbot-distilbert
from google.colab import files; files.download("maintainerbot-distilbert.zip")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/1883 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'method': 'Fine-tuned DistilBERT', 'accuracy': 0.6144, 'macro_f1': 0.6682, 'sec_per_issue': 0.002, 'n': 682}
              precision    recall  f1-score   support

         bug       0.40      0.89      0.56       141
     feature       0.69      0.88      0.77        75
    question       0.91      0.36      0.51       362
        docs       0.75      0.94      0.83       104

    accuracy                           0.61       682
   macro avg       0.69      0.77      0.67       682
weighted avg       0.76      0.61      0.60       682

confusion matrix (rows=true, cols=pred, order: ['bug', 'feature', 'question', 'docs'] )
[[126   3  10   2]
 [  5  66   2   2]
 [178  26 129  29]
 [  4   1   1  98]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>